<a href="https://colab.research.google.com/github/FiloEmad/Solship_ML_Hackathon/blob/main/Solship_ML_Hackathon_MCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Attempt to read the CSV, letting pandas infer the separator
ENERGY_Hackathon_df = pd.read_csv('/content/ENERGY_Hackathon_DataSet(Sheet1).csv', sep=None, engine='python')

display(ENERGY_Hackathon_df.head())

,battery_p,grid_p,load_p,pv_p,timestamp,Selling_price_eur_kwh
0,0,"3,04","3,05",0,2024-01-01 00:00:00,"0,10709"
1,0,"0,34","0,34",0,2024-01-01 00:15:00,"0,10709"
2,0,"0,37","0,37",0,2024-01-01 00:30:00,"0,10709"
3,0,"0,37","0,37",0,2024-01-01 00:45:00,"0,10709"
4,0,"2,4","2,4",0,2024-01-01 01:00:00,"0,104"


In [ ]:
import numpy as np

# Ensure 'timestamp' is datetime
ENERGY_Hackathon_df['timestamp'] = pd.to_datetime(ENERGY_Hackathon_df['timestamp'])

# Extract hour and day of week
ENERGY_Hackathon_df['hour'] = ENERGY_Hackathon_df['timestamp'].dt.hour
ENERGY_Hackathon_df['dayofweek'] = ENERGY_Hackathon_df['timestamp'].dt.dayofweek # Monday=0, Sunday=6

# Define conditions for each band
conditions = [
    # F1: Mon–Fri 08:00–19:00
    (ENERGY_Hackathon_df['dayofweek'].isin([0, 1, 2, 3, 4])) & (ENERGY_Hackathon_df['hour'] >= 8) & (ENERGY_Hackathon_df['hour'] < 19),

    # F2: Mon–Fri 07:00–08:00 and 19:00–23:00
    (ENERGY_Hackathon_df['dayofweek'].isin([0, 1, 2, 3, 4])) & (
        ((ENERGY_Hackathon_df['hour'] >= 7) & (ENERGY_Hackathon_df['hour'] < 8)) |
        ((ENERGY_Hackathon_df['hour'] >= 19) & (ENERGY_Hackathon_df['hour'] < 23))
    ),

    # F2: Saturday 07:00–23:00
    (ENERGY_Hackathon_df['dayofweek'] == 5) & (ENERGY_Hackathon_df['hour'] >= 7) & (ENERGY_Hackathon_df['hour'] < 23),

    # F3: Mon–Sat 00:00–07:00 and 23:00–24:00
    (ENERGY_Hackathon_df['dayofweek'].isin([0, 1, 2, 3, 4, 5])) & (
        ((ENERGY_Hackathon_df['hour'] >= 0) & (ENERGY_Hackathon_df['hour'] < 7)) |
        ((ENERGY_Hackathon_df['hour'] >= 23) & (ENERGY_Hackathon_df['hour'] < 24))
    ),

    # F3: Sundays: all day (National holidays are mentioned but no list was provided)
    (ENERGY_Hackathon_df['dayofweek'] == 6)
]

# Define corresponding prices
choices = [
    0.2540, # F1
    0.2682, # F2 (Mon-Fri part)
    0.2682, # F2 (Saturday part)
    0.2440, # F3 (Mon-Sat part)
    0.2440  # F3 (Sunday part)
]

# Create the new column using np.select
# A default value is set to handle any case not covered, though the conditions should cover all possibilities.
ENERGY_Hackathon_df['buying_price_eur_kwh'] = np.select(conditions, choices, default=np.nan)

# Drop temporary columns
ENERGY_Hackathon_df = ENERGY_Hackathon_df.drop(columns=['hour', 'dayofweek'])

display(ENERGY_Hackathon_df.head())

,battery_p,grid_p,load_p,pv_p,timestamp,Selling_price_eur_kwh,buying_price_eur_kwh
0,0,"3,04","3,05",0,2024-01-01 00:00:00,"0,10709",0.244
1,0,"0,34","0,34",0,2024-01-01 00:15:00,"0,10709",0.244
2,0,"0,37","0,37",0,2024-01-01 00:30:00,"0,10709",0.244
3,0,"0,37","0,37",0,2024-01-01 00:45:00,"0,10709",0.244
4,0,"2,4","2,4",0,2024-01-01 01:00:00,"0,104",0.244


In [ ]:
# import numpy as np
# import cvxpy as cp

# class BatteryDispatchController:
#     def __init__(self,
#                  battery_capacity_kwh=16.0,
#                  battery_power_max_kw=8.0,
#                  battery_efficiency=0.90,
#                  grid_limit_kw=6.0,
#                  initial_soc=0.5,
#                  dt_hours=0.25,
#                  horizon_hours=12):

#         self.capacity = battery_capacity_kwh
#         self.p_batt_max = battery_power_max_kw
#         self.eta = np.sqrt(battery_efficiency)  # 0.9487 per direction
#         self.grid_limit = grid_limit_kw
#         self.soc = initial_soc
#         self.dt = dt_hours
#         self.H = int(horizon_hours / dt_hours)  # 48 timesteps for 12 hours

#     def optimize_dispatch(self, load_forecast, pv_forecast, buy_price, sell_price):
#         """
#         Solve MPC optimization over horizon H.
#         Returns optimal battery power for first timestep only.

#         Args:
#             load_forecast: array of length H (kW)
#             pv_forecast: array of length H (kW)
#             buy_price: array of length H (€/kWh)
#             sell_price: array of length H (€/kWh)

#         Returns:
#             p_battery_optimal: float (kW), negative=charging, positive=discharging
#         """
#         H = min(self.H, len(load_forecast))

#         # Decision variables
#         p_battery = cp.Variable(H)  # Battery power (kW)
#         p_grid = cp.Variable(H)     # Grid power (kW)
#         soc = cp.Variable(H + 1)    # State of charge (0-1)

#         # Auxiliary variables for absolute value in objective
#         p_grid_import = cp.Variable(H)  # Grid import (≥0)
#         p_grid_export = cp.Variable(H)  # Grid export (≥0)

#         # Objective: minimize electricity bill
#         cost_import = cp.sum(cp.multiply(p_grid_import, buy_price[:H]) * self.dt)
#         revenue_export = cp.sum(cp.multiply(p_grid_export, sell_price[:H]) * self.dt)
#         objective = cp.Minimize(cost_import - revenue_export)

#         # Constraints
#         constraints = []

#         # Initial SoC
#         constraints.append(soc[0] == self.soc)

#         for t in range(H):
#             # Energy balance: P_grid + P_pv + P_battery = P_load
#             constraints.append(p_grid[t] + pv_forecast[t] + p_battery[t] == load_forecast[t])

#             # Grid power limits
#             constraints.append(p_grid[t] >= -self.grid_limit)
#             constraints.append(p_grid[t] <= self.grid_limit)

#             # Battery power limits
#             constraints.append(p_battery[t] >= -self.p_batt_max)
#             constraints.append(p_battery[t] <= self.p_batt_max)

#             # SoC limits
#             constraints.append(soc[t + 1] >= 0)
#             constraints.append(soc[t + 1] <= 1)

#             # SoC dynamics (linearized with efficiency)
#             # Charging: P_battery < 0, SoC increases
#             # Discharging: P_battery > 0, SoC decreases
#             # Approximation: SoC change ≈ -P_battery * dt / (eta * capacity)
#             constraints.append(
#                 soc[t + 1] == soc[t] - p_battery[t] * self.dt / (self.eta * self.capacity)
#             )

#             # Grid import/export auxiliary variables
#             constraints.append(p_grid_import[t] >= p_grid[t])
#             constraints.append(p_grid_import[t] >= 0)
#             constraints.append(p_grid_export[t] >= -p_grid[t])
#             constraints.append(p_grid_export[t] >= 0)

#         # Solve
#         problem = cp.Problem(objective, constraints)
#         problem.solve(solver=cp.ECOS, verbose=False)

#         if problem.status not in ['optimal', 'optimal_inaccurate']:
#             # Fallback: no battery action
#             return 0.0

#         # Extract first timestep decision
#         p_battery_optimal = p_battery.value[0]

#         # Update internal SoC for next call
#         soc_change = -p_battery_optimal * self.dt / (self.eta * self.capacity)
#         self.soc = np.clip(self.soc + soc_change, 0, 1)

#         return p_battery_optimal

#     def reset_soc(self, soc=0.5):
#         """Reset SoC to initial value."""
#         self.soc = soc


# def rolling_horizon_dispatch(load_forecast_full, pv_forecast_full,
#                              buy_price_full, sell_price_full,
#                              horizon_hours=12):
#     """
#     Execute rolling-horizon MPC over full year.

#     Args:
#         load_forecast_full: array (kW) for full period
#         pv_forecast_full: array (kW) for full period
#         buy_price_full: array (€/kWh) for full period
#         sell_price_full: array (€/kWh) for full period
#         horizon_hours: MPC horizon in hours

#     Returns:
#         p_battery_trajectory: array of battery power decisions (kW)
#     """
#     controller = BatteryDispatchController(horizon_hours=horizon_hours)
#     N = len(load_forecast_full)
#     p_battery_trajectory = np.zeros(N)

#     for t in range(N):
#         # Extract horizon window
#         t_end = min(t + controller.H, N)

#         load_horizon = load_forecast_full[t:t_end]
#         pv_horizon = pv_forecast_full[t:t_end]
#         buy_price_horizon = buy_price_full[t:t_end]
#         sell_price_horizon = sell_price_full[t:t_end]

#         # Optimize
#         p_batt = controller.optimize_dispatch(
#             load_horizon, pv_horizon, buy_price_horizon, sell_price_horizon
#         )

#         p_battery_trajectory[t] = p_batt

#     return p_battery_trajectory

In [ ]:
import pandas as pd
import numpy as np
import cvxpy as cp

# Install ECOS solver
!pip install ecos -qqq

# =========================================================
# BATTERY DISPATCH CONTROLLER
# =========================================================

class BatteryDispatchController:
    def __init__(self,
                 battery_capacity_kwh=16.0,
                 battery_power_max_kw=8.0,
                 battery_efficiency=0.90,
                 grid_limit_kw=6.0,
                 initial_soc=0.5,
                 dt_hours=0.25,
                 horizon_hours=12):

        self.capacity = battery_capacity_kwh
        self.p_batt_max = battery_power_max_kw
        self.eta = np.sqrt(battery_efficiency)
        self.grid_limit = grid_limit_kw
        self.soc = initial_soc
        self.dt = dt_hours
        self.H = int(horizon_hours / dt_hours)

    def optimize_dispatch(self, load_forecast, pv_forecast, buy_price, sell_price):
        H = min(self.H, len(load_forecast))

        p_battery = cp.Variable(H)
        p_grid = cp.Variable(H)
        soc = cp.Variable(H + 1)

        # Use auxiliary variables to represent import and export for DCP compliance
        p_grid_import = cp.Variable(H, nonneg=True)
        p_grid_export = cp.Variable(H, nonneg=True)

        # Objective: minimize (import cost - export revenue)
        # Using cp.multiply to avoid Matrix Multiplication UserWarnings
        cost = cp.sum(
            cp.multiply(p_grid_import, buy_price[:H]) * self.dt
            - cp.multiply(p_grid_export, sell_price[:H]) * self.dt
        )

        objective = cp.Minimize(cost)

        constraints = [soc[0] == self.soc]

        for t in range(H):
            constraints.extend([
                # Power balance
                p_grid[t] + pv_forecast[t] + p_battery[t] == load_forecast[t],
                # Split grid power into import and export components
                p_grid[t] == p_grid_import[t] - p_grid_export[t],
                # Limits
                p_grid[t] >= -self.grid_limit,
                p_grid[t] <= self.grid_limit,
                p_battery[t] >= -self.p_batt_max,
                p_battery[t] <= self.p_batt_max,
                soc[t+1] >= 0,
                soc[t+1] <= 1,
                soc[t+1] == soc[t] - p_battery[t] * self.dt / (self.eta * self.capacity)
            ])

        problem = cp.Problem(objective, constraints)
        problem.solve(solver=cp.ECOS, verbose=False)

        if problem.status not in ['optimal', 'optimal_inaccurate'] or p_battery.value is None:
            return 0.0

        p_battery_optimal = p_battery.value[0]
        # Update internal state
        self.soc = np.clip(
            self.soc - p_battery_optimal * self.dt / (self.eta * self.capacity),
            0, 1
        )

        return p_battery_optimal

# =========================================================
# LOAD DATA AND RUN SIMULATION
# =========================================================
df = ENERGY_Hackathon_df.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Filter April 2025
df = df[(df["timestamp"] >= "2025-04-01") & (df["timestamp"] < "2025-05-01")].reset_index(drop=True)

# Clean numeric columns
numeric_cols = ["load_p", "pv_p", "Selling_price_eur_kwh", "buying_price_eur_kwh"]
for col in numeric_cols:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.replace(",", ".").astype(float)

controller = BatteryDispatchController(initial_soc=0.5, horizon_hours=12)
N = len(df)
p_battery_optimized = np.zeros(N)
soc_trajectory = np.zeros(N + 1)
soc_trajectory[0] = 0.5

print(f"Processing {N} steps...")
for t in range(N):
    t_end = min(t + controller.H, N)
    p_batt = controller.optimize_dispatch(
        df["load_p"].iloc[t:t_end].values,
        df["pv_p"].iloc[t:t_end].values,
        df["buying_price_eur_kwh"].iloc[t:t_end].values,
        df["Selling_price_eur_kwh"].iloc[t:t_end].values
    )
    p_battery_optimized[t] = p_batt
    soc_trajectory[t+1] = controller.soc
    if t % 500 == 0: print(f"Progress: {t}/{N}")

df["p_battery_optimized"] = p_battery_optimized
df["soc_optimized"] = soc_trajectory[1:]
df["p_grid_optimized"] = df["load_p"] - df["pv_p"] - df["p_battery_optimized"]
df["import_cost"] = np.where(df["p_grid_optimized"] > 0, df["p_grid_optimized"] * df["buying_price_eur_kwh"] * 0.25, 0)
df["export_revenue"] = np.where(df["p_grid_optimized"] < 0, -df["p_grid_optimized"] * df["Selling_price_eur_kwh"] * 0.25, 0)

print(f"\nTotal April Bill: {df['import_cost'].sum() - df['export_revenue'].sum():.2f}")

Processing 2880 steps...
Progress: 0/2880
Progress: 500/2880
Progress: 1000/2880
Progress: 1500/2880
Progress: 2000/2880
Progress: 2500/2880

Total April Bill: -24.35


In [ ]:
print(df["buying_price_eur_kwh"].describe())
print(df["Selling_price_eur_kwh"].describe())

In [ ]:
print(np.sum(np.abs(df["p_battery_optimized"])))